# 04 - Pose Inference
Run the trained pose model on all videos to extract keypoint coordinates.

In [27]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"/content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior/260529.00000003"       # Input: session video folders
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"         # Input: trained model checkpoints
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"     # Output: pose prediction CSVs

CONFIDENCE_THRESHOLD = 0.5

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [28]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

/content/LightningPoseTrack
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.05 KiB | 540.00 KiB/s, done.
From https://github.com/kaarthik-balakrishnan/LightningPoseTrack
   1e3665e..edea1fa  main       -> origin/main
Updating 1e3665e..edea1fa
Fast-forward
 notebooks/04_Pose_Inference.ipynb | 124 ++++++++++++++++++++------------------
 1 file changed, 65 insertions(+), 59 deletions(-)
/content/LightningPoseTrack


In [30]:
# Install Lightning Pose + inference dependencies
!pip install --quiet "lightning-pose[all]" opencv-python pandas numpy pyarrow imageio[ffmpeg]

In [31]:
from pathlib import Path

model_dir = Path(DRIVE_MODELS) / "pose_model"
if model_dir.exists() and (model_dir / "config.yaml").exists():
    print(f"Model directory found: {model_dir}")
    print(f"  Contents: {[p.name for p in model_dir.iterdir()]}")
else:
    print(f"Model directory not found at {model_dir}. Complete notebook 03 first.")

Model directory found: /content/drive/My Drive/PigBehavior/trained_models/pose_model
  Contents: ['tb_logs', 'config.yaml', 'CollectedData_LP.csv', 'image_preds', 'train_status.json', 'predictions_pixel_error.csv', 'predictions.csv', 'predictions_pca_singleview_error.csv']


In [32]:
from src.io.video_inventory import scan_videos, parse_camera_from_filename

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos to process")
df[["filename", "session", "camera", "frame_count", "duration_min"]]

  Root path: /content/drive/Shareddrives/3R.Data/1P.SPRIND.Data/POC/Behavior/260529.00000003
  Path exists: True
  First 20 entries: ['161312-4.ASF', '161312-3.ASF', '161311-2.ASF', '161308-1.ASF', '161454-3.ASF', '161459-4.ASF', '161559-1.ASF', '161437-2.ASF', '161550-3.ASF', '161700-4.ASF', '161838-3.ASF', '161844-2.ASF', '161855-1.ASF', '161952-2.ASF', '162009-3.ASF', '162021-4.ASF', '162241-3.ASF', '162258-2.ASF', '162304-1.ASF', '162603-2.ASF']
  161308-1.ASF — OK (h264, 1920x1080, 10.2 fps, 1447f)
  161311-2.ASF — OK (h264, 1920x1080, 10.2 fps, 851f)
  161312-3.ASF — OK (h264, 1920x1080, 10.0 fps, 761f)
  161312-4.ASF — OK (h264, 1920x1080, 5.0 fps, 301f)
  161437-2.ASF — OK (h264, 1920x1080, 10.2 fps, 1775f)
  161454-3.ASF — OK (h264, 1920x1080, 10.0 fps, 602f)
  161459-4.ASF — OK (h264, 1920x1080, 5.0 fps, 531f)
  161550-3.ASF — OK (h264, 1920x1080, 10.0 fps, 1262f)
  161559-1.ASF — OK (h264, 1920x1080, 10.2 fps, 914f)
  161700-4.ASF — OK (h264, 1920x1080, 5.0 fps, 761f)
  1618

,filename,session,camera,frame_count,duration_min
0,161308-1.ASF,260529.00000003,1,1447,2.35
1,161311-2.ASF,260529.00000003,2,851,1.38
2,161312-3.ASF,260529.00000003,3,761,1.27
3,161312-4.ASF,260529.00000003,4,301,1.00
4,161437-2.ASF,260529.00000003,2,1775,2.89
...,...,...,...,...,...
83,220034-2.ASF,260529.00000003,2,617,1.00
84,220038-3.ASF,260529.00000003,3,602,1.00
85,220044-4.ASF,260529.00000003,4,891,2.97
86,220344-4.ASF,260529.00000003,4,666,2.22


In [33]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

from lightning_pose.api import Model

# Load the trained model from the model directory (contains config.yaml + .ckpt)
model_dir = Path(DRIVE_MODELS) / "pose_model"
model = Model.from_dir(str(model_dir))
print(f"Model loaded from {model_dir}")

pose_output_dir = Path(DRIVE_POSE_OUTPUTS)
pose_output_dir.mkdir(parents=True, exist_ok=True)

KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    session = row["session"]
    camera = row["camera"]
    stem = Path(row["filename"]).stem

    out_file = pose_output_dir / session / f"{stem}_cam{camera}_pose.parquet"
    if out_file.exists():
        print(f"Skipping {out_file.name} (already exists)")
        continue

    # Use OpenCV to read frames (DALI cannot handle .asf)
    cap = cv2.VideoCapture(str(video_path))
    kp_list, conf_list = [], []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = model.predict_frame(frame_rgb)
        kp_list.append(result["keypoints"].ravel())
        conf_list.append(result["confidence"])
    cap.release()

    # Build DataFrame matching the same column schema as predict_on_video_file output
    n_kp = len(KEYPOINT_NAMES)
    gen_df = pd.DataFrame({
        "frame": range(len(kp_list)),
        **{f"{kp}_x": [k[i*2] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_y": [k[i*2+1] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_likelihood": [c[i] for c in conf_list] for i, kp in enumerate(KEYPOINT_NAMES)},
    })
    out_file.parent.mkdir(parents=True, exist_ok=True)
    gen_df.to_parquet(str(out_file))
    print(f"Saved: {out_file} ({len(gen_df)} frames)")

print("\nPose inference complete!")

Model loaded from /content/drive/My Drive/PigBehavior/trained_models/pose_model


Processing videos:   0%|          | 0/88 [00:00<?, ?it/s]

	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL omegaconf.dictconfig.DictConfig was not an allowed global by default. Please use `torch.serialization.add_safe_globals([omegaconf.dictconfig.DictConfig])` or the `torch.serialization.safe_globals([omegaconf.dictconfig.DictConfig])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.
Attempting to load with weights_only=False...

Initializi

In [34]:
# Verify outputs
output_files = list(pose_output_dir.rglob("*.parquet"))
print(f"Total pose output files: {len(output_files)}")
for f in output_files:
    print(f"  {f.relative_to(pose_output_dir)}")

Total pose output files: 88
  260529.00000003/161308-1_cam1_pose.parquet
  260529.00000003/161311-2_cam2_pose.parquet
  260529.00000003/161312-3_cam3_pose.parquet
  260529.00000003/161312-4_cam4_pose.parquet
  260529.00000003/161437-2_cam2_pose.parquet
  260529.00000003/161454-3_cam3_pose.parquet
  260529.00000003/161459-4_cam4_pose.parquet
  260529.00000003/161550-3_cam3_pose.parquet
  260529.00000003/161559-1_cam1_pose.parquet
  260529.00000003/161700-4_cam4_pose.parquet
  260529.00000003/161838-3_cam3_pose.parquet
  260529.00000003/161844-2_cam2_pose.parquet
  260529.00000003/161855-1_cam1_pose.parquet
  260529.00000003/161952-2_cam2_pose.parquet
  260529.00000003/162009-3_cam3_pose.parquet
  260529.00000003/162021-4_cam4_pose.parquet
  260529.00000003/162241-3_cam3_pose.parquet
  260529.00000003/162258-2_cam2_pose.parquet
  260529.00000003/162304-1_cam1_pose.parquet
  260529.00000003/162603-2_cam2_pose.parquet
  260529.00000003/162610-3_cam3_pose.parquet
  260529.00000003/162631-4_